# 00 — data peek

**Exploration only. This notebook is never part of the pipeline.**

Anything here that turns out to matter moves into the package and gets a test:

| what you found | where it belongs |
| --- | --- |
| a loading quirk | `afl/data/loaders.py` |
| a feature idea | `afl/defend/features.py` |
| an attack shape | `afl/attack/engines/` + a row in `vectors.yaml` |
| a distribution check | `afl/fidelity/level1_statistical.py` |

A number that only exists in a notebook is not a result — nothing here is reproducible from `(config, seed)`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from afl.attack.simulator import Simulator
from afl.attack.templates import registry
from afl.data.loaders import to_frame
from afl.utils.seed import set_all_seeds

set_all_seeds(1337)
pd.set_option("display.width", 140)

## The vectors on offer

Swap the simulator for `afl.data.loaders.load("paysim")` once the raw file is in `data/raw/`.

In [ ]:
pd.DataFrame(
    [
        {
            "id": v.vector_id,
            "name": v.name,
            "engine": v.engine,
            "maturity": v.maturity,
            "why": v.why,
        }
        for v in registry.list_vectors()
    ]
).set_index("id")

In [ ]:
sim = Simulator(seed=1337, n_background=3000, n_episodes=4)
batch = sim.generate(registry.get("S1").to_attack_params())
df = to_frame(batch.transactions)

print(f"{len(df)} rows, {df.is_fraud.sum()} fraud ({df.is_fraud.mean():.2%})")
df.head()

## Amounts by label

If fraud is separable on amount alone, the vector is too easy and the recall number that follows means nothing.

In [ ]:
df.groupby("is_fraud").amount.describe()[["count", "mean", "50%", "max"]]

In [ ]:
ax = df[~df.is_fraud].amount.clip(upper=5000).plot.hist(bins=60, alpha=0.6, label="legit")
df[df.is_fraud].amount.clip(upper=5000).plot.hist(bins=60, alpha=0.6, ax=ax, label="fraud")
ax.legend()
ax.set_xlabel("amount (clipped at 5k)");

## Pacing

Inter-arrival per sender. Real accounts are bursty; a generator that emits Poisson traffic shows up here as a flat gap distribution.

In [ ]:
gaps = df.sort_values("ts").groupby("src").ts.diff().dt.total_seconds().dropna()
print(f"median gap {gaps.median():,.0f}s · CV {gaps.std() / gaps.mean():.2f}  (>1 = bursty)")
gaps.clip(upper=86_400).plot.hist(bins=60);

## Graph shape

Fan-in on the beneficiary side is the S1 tell. Compare the fraud subgraph against the background.

In [ ]:
for label, part in (("legit", df[~df.is_fraud]), ("fraud", df[df.is_fraud])):
    in_deg = part.groupby("dst").size()
    print(
        f"{label:6s} beneficiaries={len(in_deg):4d}  max in-degree={in_deg.max():3d}  "
        f"top-1 share={in_deg.max() / len(part):.2%}"
    )

## Where this goes next

- A tell you can see in these cells is a feature — add it to `features.py` **with a causality test**.
- A tell that is *too* visible is a realism problem — tighten the bounds in `vectors.yaml`, or add a check in `realism.py`.
- Either way, run `make smoke` before pushing. Nothing in this notebook is checked by anything.